# 🔬 HeatDraft ML Pipeline — Full Version

**Features:**
- ✅ Automatic file upload (CSV / XLSX)
- ✅ **Feature similarity dropping** (correlation + VIF + mutual information)
- ✅ **Masking** (target masking, feature masking, row masking)
- ✅ RDKit molecular descriptors (optional)
- ✅ Optuna hyperparameter tuning (XGBoost, ExtraTrees, RandomForest, HistGB)
- ✅ Dynamic sample weighting for high-performance zone
- ✅ Stacking ensemble
- ✅ Comprehensive diagnostics & plots
- ✅ One-click result download

---
**Run cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8**

## 📦 Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'optuna', 'xgboost', 'scikit-learn', 'rdkit',
    'matplotlib', 'seaborn', 'pandas', 'numpy',
    'openpyxl', 'statsmodels', 'scipy'
]

print('Installing packages...')
for pkg in packages:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', pkg],
        capture_output=True, text=True
    )
    print(f'  {"✅" if r.returncode == 0 else "❌"} {pkg}')

print('\n✅ Done. Restart runtime if prompted, then re-run from Cell 2.')

## 📁 Cell 2 — Upload Data File

In [ ]:
from google.colab import files
import pandas as pd
from IPython.display import display, HTML

print('📂 Upload your .csv or .xlsx data file:')
uploaded = files.upload()

if not uploaded:
    raise ValueError('No file uploaded!')

UPLOADED_FILENAME = list(uploaded.keys())[0]
with open(UPLOADED_FILENAME, 'wb') as f:
    f.write(uploaded[UPLOADED_FILENAME])

try:
    _preview = pd.read_excel(UPLOADED_FILENAME) if UPLOADED_FILENAME.endswith('.xlsx') else pd.read_csv(UPLOADED_FILENAME)
    display(HTML(f'<b>✅ Uploaded:</b> <code>{UPLOADED_FILENAME}</code> — {_preview.shape[0]} rows × {_preview.shape[1]} cols'))
    display(_preview.head(3))
    print('\nAll columns:')
    print(list(_preview.columns))
except Exception as e:
    print(f'Preview failed: {e}. File saved as: {UPLOADED_FILENAME}')

## ⚙️ Cell 3 — Configure Parameters

> **Edit all values here before running the pipeline.**

In [ ]:
import os

# ═══════════════════════════════════════════════════════════════
#  SECTION A: BASIC SETTINGS
# ═══════════════════════════════════════════════════════════════
TARGET_COLUMN    = 'Removal rate (%)'   # Column to predict
HIGH_THRESHOLD   = 90.0                 # Defines "high performance" zone
TEST_SIZE        = 0.2                  # Holdout fraction
OPTUNA_TRIALS    = 60                   # Total Optuna trials (split across models)
RANDOM_SEED      = 42
OUTPUT_DIR       = 'outputs'

# ═══════════════════════════════════════════════════════════════
#  SECTION B: FEATURE SIMILARITY DROPPING
# ═══════════════════════════════════════════════════════════════
ENABLE_FEATURE_DROPPING  = True         # Master switch

# --- Pearson / Spearman correlation threshold ---
# Features with |corr| >= threshold are considered redundant.
# Lower = more aggressive dropping. Range: 0.70 – 0.99
CORR_THRESHOLD           = 0.92
CORR_METHOD              = 'pearson'    # 'pearson' | 'spearman' | 'both'

# --- Variance Inflation Factor (VIF) ---
# Drop features whose VIF exceeds this value (multicollinearity).
# Common thresholds: 5 (strict) / 10 (moderate) / 20 (lenient)
ENABLE_VIF_DROPPING      = True
VIF_THRESHOLD            = 10.0

# --- Mutual Information (MI) similarity ---
# Drops features with MI similarity above threshold relative to
# the most informative feature in each correlated cluster.
ENABLE_MI_CLUSTER_DROP   = True
MI_CORR_FLOOR            = 0.85         # Correlation floor to form MI clusters

# ═══════════════════════════════════════════════════════════════
#  SECTION C: MASKING
# ═══════════════════════════════════════════════════════════════
# Target masking — replace target values in a range with NaN
# (those rows are excluded from training but kept for diagnostics)
ENABLE_TARGET_MASK       = False
TARGET_MASK_RANGE        = (40.0, 60.0)  # (low, high) — values INSIDE are masked

# Feature masking — set specific feature values to NaN
# Use when certain feature × condition combinations are unreliable.
# Format: { 'ColumnName': ('condition_col', operator, threshold) }
# Example: mask 'pH' when 'Temperature' > 80
ENABLE_FEATURE_MASK      = False
FEATURE_MASK_RULES       = {
    # 'pH': ('Temperature', '>', 80),
    # 'Dose (mg/L)': ('Time (min)', '<', 10),
}

# Row masking — drop rows matching conditions
# Format: list of (column, operator, value)
ENABLE_ROW_MASK          = False
ROW_MASK_CONDITIONS      = [
    # ('pH', '<', 2.0),
    # ('Temperature', '>', 100),
]

# ═══════════════════════════════════════════════════════════════
#  SECTION D: ADVANCED
# ═══════════════════════════════════════════════════════════════
NEAR_ZERO_VAR_THRESHOLD  = 0.01         # Drop features with variance below this
SPARSITY_THRESHOLD       = 0.05         # Drop features with < 5% non-null values
MAX_MODELS               = 4            # Models to tune: xgboost, extra_trees, random_forest, hist_gb

# ───────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration saved:')
print(f'   Target column    : {TARGET_COLUMN}')
print(f'   High threshold   : {HIGH_THRESHOLD}%')
print(f'   Test split       : {TEST_SIZE*100:.0f}%')
print(f'   Optuna trials    : {OPTUNA_TRIALS}')
print(f'   Feature dropping : {"ON" if ENABLE_FEATURE_DROPPING else "OFF"}')
print(f'     Corr threshold : {CORR_THRESHOLD} ({CORR_METHOD})')
print(f'     VIF dropping   : {"ON" if ENABLE_VIF_DROPPING else "OFF"} (threshold={VIF_THRESHOLD})')
print(f'     MI clustering  : {"ON" if ENABLE_MI_CLUSTER_DROP else "OFF"}')
print(f'   Target masking   : {"ON" if ENABLE_TARGET_MASK else "OFF"}')
print(f'   Feature masking  : {"ON" if ENABLE_FEATURE_MASK else "OFF"}')
print(f'   Row masking      : {"ON" if ENABLE_ROW_MASK else "OFF"}')

## 🔧 Cell 4 — Core Pipeline Functions

In [ ]:
import json, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesRegressor, RandomForestRegressor,
    StackingRegressor, HistGradientBoostingRegressor
)
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    STATSMODELS_OK = True
except ImportError:
    STATSMODELS_OK = False

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    RDKIT_OK = True
except ImportError:
    Chem = Descriptors = None
    RDKIT_OK = False

# ═══════════════════════════════════════════════════════════════════
#  DATA LOADING HELPERS
# ═══════════════════════════════════════════════════════════════════

def normalize_col_name(name: str) -> str:
    s = str(name).strip()
    if any(tok in s for tok in ['Ã','Â','â','Î','ð']):
        try:
            s = s.encode('latin1').decode('utf-8')
        except Exception:
            pass
    for k, v in {'\u2009':' ','\u202f':' ','\xa0':' '}.items():
        s = s.replace(k, v)
    return ' '.join(s.split())


def load_dataframe(path: str, target_hint: str) -> pd.DataFrame:
    path = Path(path)
    loaders = (
        [lambda: pd.read_excel(path, header=1), lambda: pd.read_excel(path, header=0)]
        if str(path).endswith('.xlsx') else
        [lambda: pd.read_csv(path, header=1), lambda: pd.read_csv(path, header=0)]
    )
    for load in loaders:
        try:
            df = load()
            df.columns = [normalize_col_name(c) for c in df.columns]
            if infer_target_column(df.columns.tolist(), target_hint):
                return df
        except Exception:
            continue
    raise RuntimeError('Failed to load dataset with header=0 or header=1.')


def infer_target_column(columns: List[str], target_hint: str) -> Optional[str]:
    if target_hint in columns:
        return target_hint
    low = {c.lower(): c for c in columns}
    if target_hint.lower() in low:
        return low[target_hint.lower()]
    for c in columns:
        if 'removal' in c.lower() and '%' in c:
            return c
    return None


def add_rdkit_descriptors(df: pd.DataFrame) -> pd.DataFrame:
    smiles_col = next((c for c in df.columns if c.lower() == 'smiles'), None)
    if smiles_col is None or not RDKIT_OK:
        return df
    fns = {
        'RD_MW': Descriptors.MolWt, 'RD_LogP': Descriptors.MolLogP,
        'RD_TPSA': Descriptors.TPSA, 'RD_HBA': Descriptors.NumHAcceptors,
        'RD_HBD': Descriptors.NumHDonors, 'RD_Rings': Descriptors.RingCount,
        'RD_RotBonds': Descriptors.NumRotatableBonds,
        'RD_FracCSP3': Descriptors.FractionCSP3,
    }
    mols = df[smiles_col].apply(
        lambda x: Chem.MolFromSmiles(x) if isinstance(x, str) and x.strip() else None
    )
    for feat, fn in fns.items():
        df[feat] = mols.apply(lambda m: fn(m) if m is not None else np.nan)
    print(f'  RDKit: added 8 descriptors from "{smiles_col}" '
          f'(valid SMILES: {mols.notna().mean()*100:.1f}%)')
    return df


# ═══════════════════════════════════════════════════════════════════
#  MASKING
# ═══════════════════════════════════════════════════════════════════

MASK_OPS = {
    '>':  lambda col, v: col > v,
    '<':  lambda col, v: col < v,
    '>=': lambda col, v: col >= v,
    '<=': lambda col, v: col <= v,
    '==': lambda col, v: col == v,
    '!=': lambda col, v: col != v,
}


def apply_target_mask(df: pd.DataFrame, target_col: str,
                      mask_range: Tuple[float, float]) -> Tuple[pd.DataFrame, pd.Index]:
    """
    Mask (set to NaN) target values that fall INSIDE [low, high].
    Returns the modified dataframe and the index of masked rows.
    """
    low, high = mask_range
    mask = (df[target_col] >= low) & (df[target_col] <= high)
    masked_idx = df.index[mask]
    df = df.copy()
    df.loc[mask, target_col] = np.nan
    print(f'  [Target Mask] Masked {mask.sum()} rows where '
          f'{target_col} ∈ [{low}, {high}]')
    return df, masked_idx


def apply_feature_masks(df: pd.DataFrame,
                        rules: Dict[str, Tuple]) -> Tuple[pd.DataFrame, Dict]:
    """
    For each rule (feature_col: (condition_col, op, threshold)),
    set feature_col to NaN where condition holds.
    Returns modified df and a report dict.
    """
    df = df.copy()
    report = {}
    for feat_col, (cond_col, op, threshold) in rules.items():
        if feat_col not in df.columns:
            print(f'  [Feature Mask] WARNING: "{feat_col}" not in columns — skipped.')
            continue
        if cond_col not in df.columns:
            print(f'  [Feature Mask] WARNING: condition col "{cond_col}" not found — skipped.')
            continue
        op_fn = MASK_OPS.get(op)
        if op_fn is None:
            raise ValueError(f'Unknown operator "{op}". Use one of {list(MASK_OPS)}')
        cond = op_fn(df[cond_col], threshold)
        count = int(cond.sum())
        df.loc[cond, feat_col] = np.nan
        report[feat_col] = {
            'condition': f'{cond_col} {op} {threshold}',
            'masked_cells': count
        }
        print(f'  [Feature Mask] "{feat_col}" → NaN where {cond_col} {op} {threshold} '
              f'({count} cells masked)')
    return df, report


def apply_row_masks(df: pd.DataFrame,
                    conditions: List[Tuple]) -> Tuple[pd.DataFrame, int]:
    """
    Drop rows matching ANY of the conditions.
    conditions: list of (col, op, value)
    """
    df = df.copy()
    combined_mask = pd.Series(False, index=df.index)
    for col, op, val in conditions:
        if col not in df.columns:
            print(f'  [Row Mask] WARNING: "{col}" not in columns — skipped.')
            continue
        op_fn = MASK_OPS.get(op)
        if op_fn is None:
            raise ValueError(f'Unknown operator "{op}".')
        combined_mask |= op_fn(df[col], val)
        print(f'  [Row Mask] Rule: {col} {op} {val} — '
              f'{op_fn(df[col], val).sum()} rows flagged')
    n_dropped = int(combined_mask.sum())
    df = df.loc[~combined_mask].copy()
    print(f'  [Row Mask] Total rows dropped: {n_dropped}')
    return df, n_dropped


# ═══════════════════════════════════════════════════════════════════
#  FEATURE SIMILARITY DROPPING
# ═══════════════════════════════════════════════════════════════════

def drop_near_zero_variance(X: pd.DataFrame, threshold: float) -> Tuple[pd.DataFrame, List[str]]:
    """Drop numeric features with variance below threshold (after scaling)."""
    num = X.select_dtypes(include=[np.number])
    # Normalize before variance check to avoid unit sensitivity
    scaled = (num - num.mean()) / (num.std().replace(0, 1))
    var = scaled.var()
    dropped = var[var < threshold].index.tolist()
    if dropped:
        print(f'  [NZV] Dropped {len(dropped)} near-zero-variance features: {dropped}')
    return X.drop(columns=dropped, errors='ignore'), dropped


def build_correlation_clusters(X: pd.DataFrame, threshold: float,
                                method: str) -> List[List[str]]:
    """
    Build clusters of highly correlated features using greedy grouping.
    Each cluster is a list of feature names.
    """
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if len(num_cols) < 2:
        return []

    X_filled = X[num_cols].fillna(X[num_cols].median())

    if method == 'spearman':
        corr = X_filled.rank().corr(method='pearson').abs()
    elif method == 'both':
        c1 = X_filled.corr(method='pearson').abs()
        c2 = X_filled.rank().corr(method='pearson').abs()
        corr = (c1 + c2) / 2
    else:  # pearson
        corr = X_filled.corr(method='pearson').abs()

    visited = set()
    clusters = []
    for col in num_cols:
        if col in visited:
            continue
        similar = corr[col][corr[col] >= threshold].index.tolist()
        similar = [c for c in similar if c != col]
        if similar:
            cluster = [col] + [c for c in similar if c not in visited]
            visited.update(cluster)
            clusters.append(cluster)
        else:
            visited.add(col)
    return clusters


def select_cluster_representative(cluster: List[str], X: pd.DataFrame,
                                   y: pd.Series, method: str) -> str:
    """
    From a cluster of correlated features, pick the most informative one.
    Methods: 'mi' (mutual information), 'pearson' (correlation with target),
             'spearman', 'variance'
    """
    num_cols = [c for c in cluster if c in X.select_dtypes(include=[np.number]).columns]
    if not num_cols:
        return cluster[0]

    X_sub = X[num_cols].fillna(X[num_cols].median())
    y_clean = y.fillna(y.median())

    if method == 'mi':
        scores = mutual_info_regression(
            X_sub.values, y_clean.values, random_state=42
        )
        return num_cols[int(np.argmax(scores))]

    if method in ('pearson', 'spearman'):
        corrs = [abs(X_sub[c].corr(y_clean, method=method)) for c in num_cols]
        return num_cols[int(np.argmax(corrs))]

    # fallback: highest variance
    return X_sub.std().idxmax()


def drop_correlated_features(
    X: pd.DataFrame, y: pd.Series,
    corr_threshold: float, corr_method: str,
    use_mi: bool, mi_corr_floor: float
) -> Tuple[pd.DataFrame, Dict]:
    """
    Full feature-similarity dropping pipeline:
      1. Build correlation clusters at corr_threshold
      2. Within each cluster, keep most informative feature (by MI or correlation)
      3. Optionally form finer MI-based clusters at mi_corr_floor
    Returns reduced X and a detailed report.
    """
    report = {
        'original_features': X.shape[1],
        'clusters': [],
        'dropped': [],
        'kept': [],
    }

    clusters = build_correlation_clusters(X, corr_threshold, corr_method)

    to_drop = set()
    for cluster in clusters:
        rep = select_cluster_representative(cluster, X, y, 'mi' if use_mi else corr_method)
        drop_in_cluster = [c for c in cluster if c != rep]
        to_drop.update(drop_in_cluster)
        report['clusters'].append({
            'kept': rep,
            'dropped': drop_in_cluster,
            'size': len(cluster)
        })

    if use_mi and mi_corr_floor < corr_threshold:
        mi_clusters = build_correlation_clusters(X, mi_corr_floor, corr_method)
        for cluster in mi_clusters:
            remaining = [c for c in cluster if c not in to_drop]
            if len(remaining) < 2:
                continue
            rep = select_cluster_representative(remaining, X, y, 'mi')
            for c in remaining:
                if c != rep:
                    to_drop.add(c)

    X_reduced = X.drop(columns=list(to_drop), errors='ignore')
    report['dropped'] = sorted(to_drop)
    report['kept'] = sorted(X_reduced.columns.tolist())
    report['final_features'] = X_reduced.shape[1]
    report['n_dropped'] = len(to_drop)
    return X_reduced, report


def drop_high_vif_features(
    X: pd.DataFrame, vif_threshold: float
) -> Tuple[pd.DataFrame, List[str], pd.DataFrame]:
    """
    Iteratively drop the feature with the highest VIF until all VIFs
    are below vif_threshold. Returns reduced X, dropped feature list,
    and the final VIF table.
    """
    if not STATSMODELS_OK:
        print('  [VIF] statsmodels not available — skipping VIF dropping.')
        return X, [], pd.DataFrame()

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    X_num = X[num_cols].fillna(X[num_cols].median())
    # Drop constant cols to avoid division by zero in VIF
    X_num = X_num.loc[:, X_num.std() > 0]
    cols = X_num.columns.tolist()
    dropped_vif = []

    while True:
        if len(cols) < 2:
            break
        data = X_num[cols].values
        vif_values = [
            variance_inflation_factor(data, i)
            for i in range(data.shape[1])
        ]
        max_vif = max(vif_values)
        if max_vif <= vif_threshold:
            break
        worst = cols[int(np.argmax(vif_values))]
        print(f'  [VIF] Dropping "{worst}" (VIF={max_vif:.2f} > {vif_threshold})')
        dropped_vif.append(worst)
        cols.remove(worst)

    vif_data = []
    if len(cols) >= 2:
        data = X_num[cols].values
        for i, c in enumerate(cols):
            try:
                vif_data.append({'feature': c, 'VIF': round(variance_inflation_factor(data, i), 3)})
            except Exception:
                vif_data.append({'feature': c, 'VIF': np.nan})

    vif_table = pd.DataFrame(vif_data).sort_values('VIF', ascending=False).reset_index(drop=True)
    X_out = X.drop(columns=dropped_vif, errors='ignore')
    return X_out, dropped_vif, vif_table


def full_feature_selection(
    X: pd.DataFrame, y: pd.Series,
    nzv_threshold: float,
    sparsity_threshold: float,
    corr_threshold: float,
    corr_method: str,
    enable_vif: bool,
    vif_threshold: float,
    enable_mi: bool,
    mi_corr_floor: float,
) -> Tuple[pd.DataFrame, Dict]:
    """
    Orchestrates the complete feature selection pipeline:
      Step 1: Drop sparse features (< sparsity_threshold non-null)
      Step 2: Drop near-zero-variance features
      Step 3: Drop correlated features (Pearson/Spearman clusters)
      Step 4: Drop high-VIF features (multicollinearity)
    """
    full_report = {'steps': {}}
    n_start = X.shape[1]

    # Step 1: Sparsity
    sparse_dropped = [c for c in X.columns if X[c].notna().mean() < sparsity_threshold]
    X = X.drop(columns=sparse_dropped, errors='ignore')
    full_report['steps']['sparsity'] = {
        'dropped': sparse_dropped, 'remaining': X.shape[1]
    }
    if sparse_dropped:
        print(f'  [Sparsity] Dropped {len(sparse_dropped)} sparse features: {sparse_dropped}')

    # Step 2: Near-zero variance
    X, nzv_dropped = drop_near_zero_variance(X, nzv_threshold)
    full_report['steps']['near_zero_variance'] = {
        'dropped': nzv_dropped, 'remaining': X.shape[1]
    }

    # Step 3: Correlation clusters
    print(f'  [Corr] Building correlation clusters (threshold={corr_threshold}, method={corr_method})...')
    X, corr_report = drop_correlated_features(
        X, y, corr_threshold, corr_method, enable_mi, mi_corr_floor
    )
    full_report['steps']['correlation'] = corr_report
    print(f'  [Corr] {corr_report["n_dropped"]} features dropped, '
          f'{corr_report["final_features"]} remaining')

    # Step 4: VIF
    vif_table = pd.DataFrame()
    if enable_vif:
        print(f'  [VIF] Running VIF analysis (threshold={vif_threshold})...')
        X, vif_dropped, vif_table = drop_high_vif_features(X, vif_threshold)
        full_report['steps']['vif'] = {
            'dropped': vif_dropped, 'remaining': X.shape[1]
        }
        print(f'  [VIF] {len(vif_dropped)} features dropped, {X.shape[1]} remaining')

    full_report['n_start'] = n_start
    full_report['n_final'] = X.shape[1]
    full_report['total_dropped'] = n_start - X.shape[1]
    full_report['vif_table'] = vif_table
    return X, full_report


# ═══════════════════════════════════════════════════════════════════
#  PREPROCESSING & MODEL TUNING
# ═══════════════════════════════════════════════════════════════════

def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]
    num_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', RobustScaler())
    ])
    cat_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, min_frequency=0.01))
    ])
    return ColumnTransformer(
        [('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)],
        remainder='drop'
    )


def model_search_space(name: str, trial: optuna.Trial, seed: int):
    if name == 'xgboost':
        return xgb.XGBRegressor(
            objective='reg:absoluteerror',
            n_estimators=trial.suggest_int('n_estimators', 200, 1200),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            min_child_weight=trial.suggest_float('min_child_weight', 1.0, 15.0),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            gamma=trial.suggest_float('gamma', 0.0, 5.0),
            random_state=seed, n_jobs=-1, tree_method='hist',
        )
    if name == 'extra_trees':
        return ExtraTreesRegressor(
            n_estimators=trial.suggest_int('n_estimators', 300, 1400),
            max_depth=trial.suggest_int('max_depth', 4, 40),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
            max_features=trial.suggest_float('max_features', 0.3, 1.0),
            criterion='absolute_error', random_state=seed, n_jobs=-1,
        )
    if name == 'random_forest':
        return RandomForestRegressor(
            n_estimators=trial.suggest_int('n_estimators', 300, 1400),
            max_depth=trial.suggest_int('max_depth', 4, 40),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
            max_features=trial.suggest_float('max_features', 0.3, 1.0),
            criterion='absolute_error', random_state=seed, n_jobs=-1,
        )
    if name == 'hist_gb':
        return HistGradientBoostingRegressor(
            loss='absolute_error',
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            max_depth=trial.suggest_int('max_depth', 3, 16),
            max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 15, 120),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 5, 60),
            l2_regularization=trial.suggest_float('l2_regularization', 1e-8, 10.0, log=True),
            random_state=seed,
        )
    raise ValueError(f'Unknown model: {name}')


def tune_model(name, X_train, y_train, preprocessor, trials, seed, high_threshold, weight_ratio):
    cv = KFold(n_splits=5, shuffle=True, random_state=seed)

    def objective(trial):
        reg = model_search_space(name, trial, seed)
        scores = []
        for tr_idx, val_idx in cv.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
            w_tr = np.where(y_tr >= high_threshold, weight_ratio, 1.0)
            Xt = preprocessor.fit_transform(X_tr, y_tr)
            Xv = preprocessor.transform(X_val)
            reg.fit(Xt, y_tr, sample_weight=w_tr)
            scores.append(mean_absolute_error(y_val, reg.predict(Xv)))
        return -np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=max(8, trials))

    best_reg = model_search_space(name, optuna.trial.FixedTrial(study.best_trial.params), seed)
    pipe = Pipeline([('prep', preprocessor), ('model', best_reg)])
    w_all = np.where(y_train >= high_threshold, weight_ratio, 1.0)
    pipe.fit(X_train, y_train, model__sample_weight=w_all)
    return pipe, study.best_trial.params


# ═══════════════════════════════════════════════════════════════════
#  EVALUATION
# ═══════════════════════════════════════════════════════════════════

def evaluate(name, model, X, y):
    p = model.predict(X)
    return {'model': name,
            'rmse': float(np.sqrt(mean_squared_error(y, p))),
            'mae': float(mean_absolute_error(y, p)),
            'r2': float(r2_score(y, p))}


def evaluate_high_focus(name, model, X, y, threshold):
    p = model.predict(X)
    return {'model': name,
            'rmse_high': float(np.sqrt(mean_squared_error(y, p))),
            'mae_high': float(mean_absolute_error(y, p)),
            'r2_high': float(r2_score(y, p)),
            'high_hit_rate': float(np.mean(p >= threshold))}


def build_low_failure_report(best_model, X_low, y_low, X_high, high_threshold):
    if len(X_low) == 0:
        return {'low_rows': 0, 'false_high_rate': float('nan'),
                'mean_overprediction': float('nan'),
                'low_rmse': float('nan'), 'low_mae': float('nan')}, pd.DataFrame()
    preds = pd.Series(best_model.predict(X_low), index=X_low.index)
    residual = preds - y_low
    fh = preds >= high_threshold
    summary = {
        'low_rows': int(len(X_low)),
        'false_high_rate': float(fh.mean()),
        'mean_overprediction': float(residual.mean()),
        'low_rmse': float(np.sqrt(mean_squared_error(y_low, preds))),
        'low_mae': float(mean_absolute_error(y_low, preds)),
    }
    num_cols = X_low.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        return summary, pd.DataFrame()
    common = [c for c in num_cols if c in X_high.columns]
    if not common:
        return summary, pd.DataFrame()
    prof = pd.DataFrame({
        'high_train_median': X_high[common].median(),
        'low_median': X_low[common].median()
    }).dropna()
    prof['abs_gap'] = (prof['low_median'] - prof['high_train_median']).abs()
    prof = prof.sort_values('abs_gap', ascending=False).head(12)
    return summary, prof.reset_index().rename(columns={'index': 'feature'})


print('✅ All pipeline functions loaded.')

## 🚀 Cell 5 — Run Full Pipeline

In [ ]:
from IPython.display import display, HTML
import ipywidgets as widgets

# ─── Logging output widget ───────────────────────────────────────────────────
out = widgets.Output()
display(out)

def log(msg='', style='', end='\n'):
    with out:
        if style:
            display(HTML(f'<span style="{style}">{msg}</span>'))
        else:
            print(msg, end=end)

def section(title):
    log(f'<br><b style="color:#1565C0;font-size:15px;border-bottom:2px solid #1565C0;">'
        f'▶ {title}</b>', style='display:block;padding:4px 0;')

# ════════════════════════════════════════════════════════════════════
#  STEP 1: LOAD DATA
# ════════════════════════════════════════════════════════════════════
section('Step 1: Loading Data')

df = load_dataframe(UPLOADED_FILENAME, TARGET_COLUMN)
df = add_rdkit_descriptors(df)

target_col = infer_target_column(df.columns.tolist(), TARGET_COLUMN)
if not target_col:
    raise ValueError(f'Target "{TARGET_COLUMN}" not found. Columns: {df.columns.tolist()[:15]}')
log(f'  Target column found: "{target_col}"')
log(f'  Raw shape: {df.shape}')

# ════════════════════════════════════════════════════════════════════
#  STEP 2: MASKING
# ════════════════════════════════════════════════════════════════════
section('Step 2: Applying Masks')

masking_report = {}

# Row masking first (drops rows entirely)
if ENABLE_ROW_MASK and ROW_MASK_CONDITIONS:
    df, n_row_dropped = apply_row_masks(df, ROW_MASK_CONDITIONS)
    masking_report['row_mask'] = {'rows_dropped': n_row_dropped}
    log(f'  After row masking: {df.shape}')
else:
    log('  Row masking: OFF')

# Feature masking (sets cells to NaN)
if ENABLE_FEATURE_MASK and FEATURE_MASK_RULES:
    df, feat_mask_report = apply_feature_masks(df, FEATURE_MASK_RULES)
    masking_report['feature_mask'] = feat_mask_report
else:
    log('  Feature masking: OFF')

# Target masking (sets target to NaN → rows excluded from training)
masked_target_idx = pd.Index([])
if ENABLE_TARGET_MASK:
    df, masked_target_idx = apply_target_mask(df, target_col, TARGET_MASK_RANGE)
    masking_report['target_mask'] = {
        'range': TARGET_MASK_RANGE,
        'masked_rows': len(masked_target_idx)
    }
else:
    log('  Target masking: OFF')

# ════════════════════════════════════════════════════════════════════
#  STEP 3: PREPARE X, y
# ════════════════════════════════════════════════════════════════════
section('Step 3: Preparing Features & Target')

df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
df = df.dropna(subset=[target_col]).copy()  # drops masked-target rows too

drop_like = ['number', 'index', 'unnamed: 0']
leak_cols = [c for c in df.columns if c.lower() in drop_like]
X_raw = df.drop(columns=[target_col] + leak_cols, errors='ignore')
y = df[target_col]

log(f'  Rows after cleaning: {len(df)}')
log(f'  Raw features: {X_raw.shape[1]}')

# ════════════════════════════════════════════════════════════════════
#  STEP 4: FEATURE SELECTION
# ════════════════════════════════════════════════════════════════════
section('Step 4: Feature Similarity Dropping')

feature_selection_report = {}

if ENABLE_FEATURE_DROPPING:
    log('  Running feature selection pipeline...')
    X, feature_selection_report = full_feature_selection(
        X=X_raw,
        y=y,
        nzv_threshold=NEAR_ZERO_VAR_THRESHOLD,
        sparsity_threshold=SPARSITY_THRESHOLD,
        corr_threshold=CORR_THRESHOLD,
        corr_method=CORR_METHOD,
        enable_vif=ENABLE_VIF_DROPPING,
        vif_threshold=VIF_THRESHOLD,
        enable_mi=ENABLE_MI_CLUSTER_DROP,
        mi_corr_floor=MI_CORR_FLOOR,
    )
    log(f'  ✅ Features: {feature_selection_report["n_start"]} → {feature_selection_report["n_final"]} '
        f'(dropped {feature_selection_report["total_dropped"]})')

    # Show corr cluster report
    corr_clusters = feature_selection_report.get('steps', {}).get('correlation', {}).get('clusters', [])
    if corr_clusters:
        log(f'  Correlation clusters found: {len(corr_clusters)}')
        cluster_rows = []
        for cl in corr_clusters:
            cluster_rows.append({
                'Kept': cl['kept'],
                'Dropped (redundant)': ', '.join(cl['dropped']) if cl['dropped'] else '—',
                'Cluster size': cl['size']
            })
        with out:
            display(pd.DataFrame(cluster_rows))

    # Show VIF table
    vif_table = feature_selection_report.get('vif_table', pd.DataFrame())
    if not vif_table.empty:
        log('  Final VIF Table (post-dropping):')
        with out:
            display(vif_table.head(20).style.background_gradient(subset=['VIF'], cmap='RdYlGn_r'))
else:
    # Basic sparsity drop only
    X = X_raw[[c for c in X_raw.columns if X_raw[c].notna().mean() >= SPARSITY_THRESHOLD]]
    log(f'  Feature dropping: OFF (kept {X.shape[1]} features)')

# ════════════════════════════════════════════════════════════════════
#  STEP 5: TRAIN / TEST SPLIT & ZONE SETUP
# ════════════════════════════════════════════════════════════════════
section('Step 5: Train/Test Split')

high_mask = y >= HIGH_THRESHOLD
X_high, y_high = X.loc[high_mask], y.loc[high_mask]
X_low, y_low   = X.loc[~high_mask], y.loc[~high_mask]

log(f'  Total rows  : {len(df)}')
log(f'  High zone   : {len(X_high)} rows (≥ {HIGH_THRESHOLD}%)')
log(f'  Low zone    : {len(X_low)} rows')
log(f'  Final feat. : {X.shape[1]}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED,
    stratify=high_mask if len(X_high) > 10 else None
)

preprocessor = build_preprocessor(X_train)

raw_ratio = len(X_low) / max(1, len(X_high))
weight_ratio = min(max(raw_ratio * 1.5, 10.0), 1000.0)
log(f'  High-sample weight ratio: {weight_ratio:.2f}')

test_high_mask = y_test >= HIGH_THRESHOLD
X_test_high = X_test[test_high_mask]
y_test_high = y_test[test_high_mask]

if len(X_test_high) == 0:
    log('  ⚠️ No high-performance rows in test set — high-zone metrics will be NaN.')

# ════════════════════════════════════════════════════════════════════
#  STEP 6: MODEL TUNING
# ════════════════════════════════════════════════════════════════════
section('Step 6: Optuna Model Tuning')

model_names = ['xgboost', 'extra_trees', 'random_forest', 'hist_gb'][:MAX_MODELS]
per_model_trials = max(8, OPTUNA_TRIALS // len(model_names))
fitted_models, best_params_report, metrics_high = {}, {}, []

log(f'  Models: {model_names}')
log(f'  Trials per model: {per_model_trials}')

for i, name in enumerate(model_names):
    log(f'\n  [{i+1}/{len(model_names)}] Tuning {name}...')
    model, params = tune_model(
        name, X_train, y_train, preprocessor,
        per_model_trials, RANDOM_SEED, HIGH_THRESHOLD, weight_ratio
    )
    fitted_models[name] = model
    best_params_report[name] = params

    g = evaluate(name, model, X_test, y_test)
    if len(X_test_high) > 0:
        h = evaluate_high_focus(name, model, X_test_high, y_test_high, HIGH_THRESHOLD)
    else:
        h = {'model': name, 'rmse_high': float('nan'),
             'mae_high': float('nan'), 'r2_high': float('nan'), 'high_hit_rate': 0.0}
    metrics_high.append({**h, 'r2_global': g['r2'], 'rmse_global': g['rmse']})
    log(f'     ✅  MAE_high={h["mae_high"]:.3f}  R²_high={h["r2_high"]:.3f}  '
        f'R²_global={g["r2"]:.3f}')

metrics_df = pd.DataFrame(metrics_high).sort_values(
    ['r2_high', 'rmse_high', 'high_hit_rate'], ascending=[False, True, False]
).reset_index(drop=True)

# ════════════════════════════════════════════════════════════════════
#  STEP 7: STACKING ENSEMBLE
# ════════════════════════════════════════════════════════════════════
section('Step 7: Stacking Ensemble')

top3 = metrics_df.head(3)['model'].tolist()
log(f'  Top-3 base learners: {top3}')

prep_stack = clone(preprocessor)
X_train_trans = prep_stack.fit_transform(X_train, y_train)

stack = StackingRegressor(
    estimators=[(n, fitted_models[n].named_steps['model']) for n in top3],
    final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 20)),
    passthrough=False, n_jobs=-1,
)
w_train = np.where(y_train >= HIGH_THRESHOLD, weight_ratio, 1.0)
stack.fit(X_train_trans, y_train, sample_weight=w_train)
final_stack_pipe = Pipeline([('prep', prep_stack), ('model', stack)])

g_s = evaluate('stacking_top3', final_stack_pipe, X_test, y_test)
if len(X_test_high) > 0:
    h_s = evaluate_high_focus('stacking_top3', final_stack_pipe, X_test_high, y_test_high, HIGH_THRESHOLD)
else:
    h_s = {'model': 'stacking_top3', 'rmse_high': float('nan'),
            'mae_high': float('nan'), 'r2_high': float('nan'), 'high_hit_rate': 0.0}
metrics_high.append({**h_s, 'r2_global': g_s['r2'], 'rmse_global': g_s['rmse']})

final_metrics = pd.DataFrame(metrics_high).sort_values(
    ['r2_high', 'rmse_high', 'high_hit_rate'], ascending=[False, True, False]
).reset_index(drop=True)

section('Final Leaderboard')
with out:
    display(final_metrics.style
            .highlight_max(subset=['r2_high', 'r2_global', 'high_hit_rate'], color='#c8e6c9')
            .highlight_min(subset=['rmse_high', 'mae_high', 'rmse_global'], color='#c8e6c9')
            .format({
                'r2_high': '{:.4f}', 'mae_high': '{:.3f}', 'rmse_high': '{:.3f}',
                'r2_global': '{:.4f}', 'rmse_global': '{:.3f}', 'high_hit_rate': '{:.3f}'
            }))

winner = final_metrics.iloc[0]['model']
best_model = final_stack_pipe if winner == 'stacking_top3' else fitted_models[winner]
log(f'<br>🥇 Best model: <b>{winner}</b>', style='color:#2E7D32;font-size:15px;')

print('\n✅ Training complete. Run Cell 6 to generate plots & reports.')

## 📊 Cell 6 — Plots & Diagnostics

In [ ]:
from IPython.display import display
import matplotlib.gridspec as gridspec

preds_plot = best_model.predict(X_test_high if len(X_test_high) > 0 else X_test)
y_plot     = y_test_high if len(X_test_high) > 0 else y_test

# ── Figure 1: 2×2 dashboard ──────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 13))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# Plot A: Predicted vs Actual
ax0 = fig.add_subplot(gs[0, 0])
ax0.scatter(y_plot, preds_plot, alpha=0.75, edgecolors='black',
            linewidths=0.25, color='#1976D2', s=60)
lo = min(float(y_plot.min()), float(preds_plot.min()))
hi = max(float(y_plot.max()), float(preds_plot.max()))
ax0.plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Ideal')
r2v = r2_score(y_plot, preds_plot) if len(y_plot) > 1 else 0
maev = mean_absolute_error(y_plot, preds_plot)
ax0.set(xlabel='Actual Removal Rate (%)', ylabel='Predicted',
        title=f'Predicted vs Actual — High Zone\n{winner}  |  R²={r2v:.3f}  MAE={maev:.2f}')
ax0.legend(fontsize=9)
ax0.grid(alpha=0.3)

# Plot B: Residuals
ax1 = fig.add_subplot(gs[0, 1])
residuals = preds_plot - y_plot.values
ax1.scatter(y_plot, residuals, alpha=0.7, edgecolors='black',
            linewidths=0.25, color='#F57C00', s=60)
ax1.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax1.set(xlabel='Actual Removal Rate (%)', ylabel='Residual (Predicted − Actual)',
        title='Residual Plot — High Zone')
ax1.grid(alpha=0.3)

# Plot C: Model comparison bar
ax2 = fig.add_subplot(gs[1, 0])
m_plot = final_metrics.dropna(subset=['r2_high']).copy()
colors = ['#2E7D32' if m == winner else '#90CAF9' for m in m_plot['model']]
bars = ax2.barh(m_plot['model'], m_plot['r2_high'],
                color=colors, edgecolor='black', linewidth=0.5)
ax2.axvline(0, color='red', linestyle='--', linewidth=0.8)
for bar, val in zip(bars, m_plot['r2_high']):
    ax2.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
ax2.set(xlabel='R² (High Zone)', title='Model R² Comparison (High Zone)')
ax2.grid(axis='x', alpha=0.3)

# Plot D: MAE comparison
ax3 = fig.add_subplot(gs[1, 1])
colors_mae = ['#2E7D32' if m == winner else '#EF9A9A' for m in m_plot['model']]
bars_mae = ax3.barh(m_plot['model'], m_plot['mae_high'],
                    color=colors_mae, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars_mae, m_plot['mae_high']):
    ax3.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=9)
ax3.set(xlabel='MAE (High Zone)', title='Model MAE Comparison (High Zone)\n(lower is better)')
ax3.grid(axis='x', alpha=0.3)

plt.suptitle(f'HeatDraft ML Pipeline — Best Model: {winner}',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig(f'{OUTPUT_DIR}/dashboard.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close()

# ── Figure 2: Correlation heatmap (before vs after feature dropping) ─────────
if ENABLE_FEATURE_DROPPING and X_raw.shape[1] <= 60:
    num_before = X_raw.select_dtypes(include=[np.number]).columns.tolist()
    num_after  = X.select_dtypes(include=[np.number]).columns.tolist()

    fig2, axes2 = plt.subplots(1, 2, figsize=(max(12, len(num_before)//2),
                                               max(9, len(num_before)//2)))
    for ax_corr, cols, title in [
        (axes2[0], num_before, f'Before Dropping ({len(num_before)} features)'),
        (axes2[1], num_after,  f'After Dropping ({len(num_after)} features)'),
    ]:
        if len(cols) > 0:
            data = X_raw[cols].fillna(X_raw[cols].median()) if 'before' in title.lower() \
                   else X[cols].fillna(X[cols].median())
            corr_mat = data.corr(method='pearson')
            mask_tri = np.triu(np.ones_like(corr_mat, dtype=bool))
            sns.heatmap(corr_mat, mask=mask_tri, ax=ax_corr, cmap='coolwarm',
                        center=0, vmin=-1, vmax=1, square=True,
                        linewidths=0.3, annot=len(cols)<=20,
                        fmt='.1f', annot_kws={'size': 7})
            ax_corr.set_title(title, fontsize=11, fontweight='bold')
            ax_corr.tick_params(axis='x', rotation=45, labelsize=7)
            ax_corr.tick_params(axis='y', rotation=0, labelsize=7)

    plt.suptitle('Pearson Correlation: Before vs After Feature Dropping',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/correlation_heatmap.png', dpi=150, bbox_inches='tight')
    display(fig2)
    plt.close()

# ── Low-zone diagnostics ─────────────────────────────────────────────────────
low_summary, low_feature_gaps = build_low_failure_report(
    best_model, X_low, y_low, X_high, HIGH_THRESHOLD
)

print('\n📉 Low-Zone Diagnostics:')
display(pd.DataFrame([low_summary]))

if not low_feature_gaps.empty:
    print('\n🔎 Top Feature Gaps (Low Zone vs High Zone):')
    display(low_feature_gaps)

    fig3, ax4 = plt.subplots(figsize=(11, 7))
    sg = low_feature_gaps.sort_values('abs_gap', ascending=True)
    yp = np.arange(len(sg))
    ax4.barh(yp - 0.2, sg['high_train_median'], height=0.4,
             label='High Zone Median', color='#4CAF50', alpha=0.85, edgecolor='black', lw=0.4)
    ax4.barh(yp + 0.2, sg['low_median'], height=0.4,
             label='Low Zone Median', color='#F44336', alpha=0.85, edgecolor='black', lw=0.4)
    ax4.set_yticks(yp)
    ax4.set_yticklabels(sg['feature'])
    ax4.set(xlabel='Median Feature Value',
            title='Feature Medians: High vs Low Performance Zones')
    ax4.legend(fontsize=10)
    ax4.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/feature_gaps.png', dpi=150, bbox_inches='tight')
    display(fig3)
    plt.close()

# ── Figure 4: MI scores (if feature dropping was on) ────────────────────────
if ENABLE_FEATURE_DROPPING:
    num_X = X.select_dtypes(include=[np.number]).fillna(X.select_dtypes(include=[np.number]).median())
    if num_X.shape[1] > 0:
        mi_scores = mutual_info_regression(num_X.values, y.values, random_state=42)
        mi_df = pd.DataFrame({'feature': num_X.columns, 'MI': mi_scores})
        mi_df = mi_df.sort_values('MI', ascending=True)

        fig4, ax5 = plt.subplots(figsize=(10, max(6, len(mi_df) * 0.35)))
        colors_mi = ['#1565C0' if s > mi_df['MI'].median() else '#90CAF9'
                     for s in mi_df['MI']]
        ax5.barh(mi_df['feature'], mi_df['MI'], color=colors_mi,
                 edgecolor='black', linewidth=0.4)
        ax5.axvline(mi_df['MI'].median(), color='red', linestyle='--',
                    linewidth=1, label='Median MI')
        ax5.set(xlabel='Mutual Information Score',
                title='Feature Importance (Mutual Information with Target)\nAfter Feature Dropping')
        ax5.legend(fontsize=9)
        ax5.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/mutual_information.png', dpi=150, bbox_inches='tight')
        display(fig4)
        plt.close()

print('\n✅ All plots generated. Run Cell 7 to save & download results.')

## 💾 Cell 7 — Save & Download Results

In [ ]:
import zipfile
from google.colab import files

# Save all reports
report = {
    'input_file': UPLOADED_FILENAME,
    'target': target_col,
    'high_threshold': HIGH_THRESHOLD,
    'rows_total': int(df.shape[0]),
    'high_rows': int(len(X_high)),
    'low_rows': int(len(X_low)),
    'features_raw': int(X_raw.shape[1]),
    'features_final': int(X.shape[1]),
    'winner': winner,
    'masking_report': masking_report,
    'feature_selection': {
        k: v for k, v in feature_selection_report.items()
        if k != 'vif_table'  # exclude df — serialized separately
    },
    'metrics': final_metrics.to_dict(orient='records'),
    'low_zone_diagnostics': low_summary,
    'best_params': best_params_report,
}

# JSON report
with open(f'{OUTPUT_DIR}/model_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

# CSV outputs
final_metrics.to_csv(f'{OUTPUT_DIR}/leaderboard.csv', index=False)
low_summary_df = pd.DataFrame([low_summary])
low_summary_df.to_csv(f'{OUTPUT_DIR}/low_zone_diagnostics.csv', index=False)

if not low_feature_gaps.empty:
    low_feature_gaps.to_csv(f'{OUTPUT_DIR}/feature_gaps.csv', index=False)

# Feature selection report
if feature_selection_report:
    fs_report_path = f'{OUTPUT_DIR}/feature_selection_report.json'
    with open(fs_report_path, 'w') as f:
        json.dump({
            k: v for k, v in feature_selection_report.items()
            if k != 'vif_table'
        }, f, indent=2, default=str)

    vif_table = feature_selection_report.get('vif_table', pd.DataFrame())
    if not vif_table.empty:
        vif_table.to_csv(f'{OUTPUT_DIR}/vif_table.csv', index=False)

# ── Zip all outputs ───────────────────────────────────────────────────────────
zip_path = 'heatdraft_results.zip'
output_files = list(Path(OUTPUT_DIR).glob('*'))

print('📦 Packaging outputs:')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in output_files:
        zf.write(fp, fp.name)
        print(f'  + {fp.name}')

print(f'\n✅ Zipped {len(output_files)} files → {zip_path}')
print('⬇️  Downloading...')
files.download(zip_path)

## 🔮 Cell 8 — (Optional) Predict on New Data

In [ ]:
from google.colab import files

print('📂 Upload a new file for prediction (must have same columns, no target needed):')
new_upload = files.upload()

if new_upload:
    new_filename = list(new_upload.keys())[0]
    with open(new_filename, 'wb') as f:
        f.write(new_upload[new_filename])

    new_df = load_dataframe(new_filename, TARGET_COLUMN)
    new_df = add_rdkit_descriptors(new_df)

    # Apply same feature masks if enabled
    if ENABLE_FEATURE_MASK and FEATURE_MASK_RULES:
        new_df, _ = apply_feature_masks(new_df, FEATURE_MASK_RULES)

    # Drop target if present
    if target_col in new_df.columns:
        new_df = new_df.drop(columns=[target_col], errors='ignore')

    # Align to trained feature set
    new_X = new_df.drop(columns=leak_cols, errors='ignore')
    missing_cols = [c for c in X.columns if c not in new_X.columns]
    for mc in missing_cols:
        new_X[mc] = np.nan  # fill missing with NaN (imputed in pipeline)
    new_X = new_X[[c for c in X.columns if c in new_X.columns]]

    preds_new = best_model.predict(new_X)
    result_df = new_df.copy()
    result_df[f'Predicted_{target_col}'] = preds_new
    result_df['High_Performance'] = preds_new >= HIGH_THRESHOLD

    pred_path = f'{OUTPUT_DIR}/predictions_{new_filename.split(".")[0]}.csv'
    result_df.to_csv(pred_path, index=False)

    print(f'\n✅ {len(result_df)} predictions saved to {pred_path}')
    print(f'   High-performance predictions: {result_df["High_Performance"].sum()} '
          f'({result_df["High_Performance"].mean()*100:.1f}%)')
    display(result_df[[f'Predicted_{target_col}', 'High_Performance']].describe())
    files.download(pred_path)
else:
    print('No file uploaded — skipping.')